# SEEA account file crawler using a country-language-aware JSON keyword configuration.

Main purpose:
- Read the 2024 Global Assessment workbook.
- Create one folder per country in seea_file/.
- Use the Website(s) hyperlinks to crawl official country websites.
- Download only files whose URL, link text, filename, or nearby text matches
  conservative SEEA account keywords generated from seea_keyword_config.json.

Important matching rule:
- A saved file must match either an exact SEEA/account phrase, or a generated
  phrase that combines an account/accounting word with a topic phrase.
- Topic-only matches such as water, energy, forest, carbon, or taxes are used
  only to help crawl pages, not as final evidence for downloading.

In [ ]:
import os
from pathlib import Path

# ------------------------------------------------------------
# 1. Set working directory and file paths
# ------------------------------------------------------------

WORKING_DIR = r"D:\School\Fall 2024\DDL\ADB\00_Revision\SEEA"

os.chdir(WORKING_DIR)

INPUT_FILE = Path("Combination.xlsx")
OUTPUT_FILE = Path("manual_validation_summary.xlsx")

print("Current working directory:", os.getcwd())



import argparse
import csv
import hashlib
import html
import json
import mimetypes
import os
import re
import time
import unicodedata
import urllib.parse
import zipfile
import xml.etree.ElementTree as ET
from dataclasses import dataclass
from heapq import heappop, heappush
from pathlib import Path

import openpyxl
import requests
from bs4 import BeautifulSoup


# ---------------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------------
# Put this script, the Excel file, and seea_keyword_config.json in the same
# working folder, or pass paths through the command line arguments below.
DEFAULT_SOURCE_XLSX = "global_assessment_for_website_2024_final.xlsx"
DEFAULT_CONFIG_JSON = "seea_keyword_config.json"
DEFAULT_SHEET_NAME = "2024 Global Assessment results"
DEFAULT_OUTPUT_DIR = "seea_file"

DOWNLOAD_LOG = "seea_file_download_log.csv"
SEARCH_LOG = "seea_file_search_log.csv"
KEYWORD_DICTIONARY_LOG = "seea_keyword_dictionary_used.csv"

DEFAULT_MAX_DEPTH = 3
DEFAULT_MAX_PAGES_PER_COUNTRY = 50
DEFAULT_MAX_LINKS_PER_PAGE = 40
DEFAULT_DELAY_SECONDS = 1.25
REQUEST_TIMEOUT = 25
MAX_DOWNLOAD_BYTES = 200_000_000
MAX_HTML_BYTES = 5_000_000

USER_AGENT = "Mozilla/5.0 (compatible; SEEA account research crawler)"

DOWNLOAD_EXTENSIONS = {
    ".csv", ".doc", ".docx", ".ods", ".pdf", ".rar",
    ".xls", ".xlsm", ".xlsx", ".zip"
}

CONTENT_TYPE_EXTENSIONS = {
    "application/pdf": ".pdf",
    "application/msword": ".doc",
    "application/vnd.ms-excel": ".xls",
    "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet": ".xlsx",
    "application/vnd.openxmlformats-officedocument.wordprocessingml.document": ".docx",
    "application/zip": ".zip",
    "application/x-zip-compressed": ".zip",
    "application/octet-stream": "",
    "text/csv": ".csv",
    "text/html": ".html",
    "application/xhtml+xml": ".html",
}

BASE_STRONG_PHRASES = {
    "seea",
    "system of environmental-economic accounting",
    "system of environmental economic accounting",
    "environmental-economic accounting",
    "environmental economic accounting",
    "environmental accounts",
    "environmental accounting",
    "environmental-economic accounts",
    "environmental economic accounts",
    "ecosystem accounts",
    "ecosystem accounting",
    "natural capital accounts",
    "natural capital accounting",
}

CRAWL_HELPER_TERMS = {
    "statistics", "data", "database", "publication", "publications", "download",
    "downloads", "table", "tables", "indicator", "indicators", "environment",
    "environmental", "resources", "resource", "account", "accounts", "accounting",
    "seea", "environmental-economic", "ecosystem", "natural capital"
}


@dataclass
class CandidateLink:
    url: str
    anchor_text: str
    nearby_text: str
    score: int
    matched_terms: list
    matched_accounts: list


# ---------------------------------------------------------------------------
# Text and URL helpers
# ---------------------------------------------------------------------------
def strip_accents(text):
    """Return a lowercase accent-insensitive version for Latin-script matching."""
    text = str(text or "").lower()
    normalized = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in normalized if not unicodedata.combining(ch))


def normalize_text(text):
    """Normalize whitespace and make text lowercase while preserving non-Latin scripts."""
    return " ".join(str(text or "").lower().split())


def compact_text(text):
    """Compact Latin text for matching phrases across punctuation or line breaks."""
    return re.sub(r"[^a-z0-9]+", "", strip_accents(text))


def clean_filename(value, max_len=100):
    value = html.unescape(str(value or "")).strip()
    value = re.sub(r"[^A-Za-z0-9._ -]+", "_", value)
    value = re.sub(r"\s+", "_", value).strip(" ._")
    return value[:max_len] or "item"


def normalize_url(url):
    if not url:
        return None

    url = html.unescape(str(url)).strip()
    if not url:
        return None

    if url.startswith("www."):
        url = "https://" + url

    if not re.match(r"^[a-zA-Z][a-zA-Z0-9+.-]*://", url):
        url = "https://" + url

    parsed = urllib.parse.urlparse(url)
    if parsed.scheme not in {"http", "https"}:
        return None

    return urllib.parse.urlunparse(parsed._replace(fragment=""))


def canonical_url(url):
    parsed = urllib.parse.urlparse(url)
    host = parsed.netloc.lower()
    path = re.sub(r"/+$", "", parsed.path or "/")
    return urllib.parse.urlunparse((parsed.scheme.lower(), host, path, "", parsed.query, ""))


def host_without_www(url):
    host = urllib.parse.urlparse(url).netloc.lower()
    return host[4:] if host.startswith("www.") else host


def is_internal_url(url, source_url):
    source_host = host_without_www(source_url)
    target_host = host_without_www(url)
    return target_host == source_host or target_host.endswith("." + source_host)


def file_extension(url, content_type=""):
    path = urllib.parse.urlparse(url).path
    ext = os.path.splitext(path)[1].lower()

    if ext and len(ext) <= 8:
        return ext

    clean_type = (content_type or "").split(";")[0].strip().lower()
    if clean_type in CONTENT_TYPE_EXTENSIONS:
        return CONTENT_TYPE_EXTENSIONS[clean_type]

    guessed = mimetypes.guess_extension(clean_type) or ""
    return guessed.lower()


def is_downloadable_url(url, content_type=""):
    ext = file_extension(url, content_type)
    if ext in DOWNLOAD_EXTENSIONS:
        return True

    clean_type = (content_type or "").split(";")[0].strip().lower()
    mapped_ext = CONTENT_TYPE_EXTENSIONS.get(clean_type, "")
    return mapped_ext in DOWNLOAD_EXTENSIONS


def unique_path(folder, filename):
    path = Path(folder) / filename
    if not path.exists():
        return str(path)

    stem, ext = os.path.splitext(filename)
    counter = 2
    while True:
        candidate = Path(folder) / f"{stem}_{counter:02d}{ext}"
        if not candidate.exists():
            return str(candidate)
        counter += 1


# ---------------------------------------------------------------------------
# JSON configuration and country-specific keyword generation
# ---------------------------------------------------------------------------
def load_keyword_config(path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def cell_is_reported(value):
    """Treat common Global Assessment positive markers as reported."""
    if value is None:
        return False
    text = str(value).strip().lower()
    return text in {"x", "1", "yes", "y", "true", "available", "reported"}


def safe_list(value):
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item).strip() for item in value if str(item).strip()]
    return [str(value).strip()]


def generate_country_keyword_profile(country, account_names, config, use_reported_accounts=True):
    """
    Build conservative keywords for one country.

    final_keywords are used to decide whether a file should be downloaded.
    They include:
      1. base SEEA phrases;
      2. exact account names in English;
      3. translated topic phrase + translated account/accounting word.

    explore_keywords are broader and used only to prioritize links while crawling.
    """
    country_languages = config.get("country_languages", {})
    account_words_by_language = config.get("account_words_by_language", {})
    topic_translations = config.get("topic_translations", {})

    languages = safe_list(country_languages.get(country)) or ["English"]
    if "English" not in languages:
        languages.append("English")

    # If the workbook has reported account columns, use only those account types.
    # If no reported account is detected, fall back to all configured account topics
    # so the crawler can still work for test runs or changed workbook formats.
    if use_reported_accounts and account_names:
        selected_accounts = [name for name in account_names if name in topic_translations]
    else:
        selected_accounts = list(topic_translations.keys())

    if not selected_accounts:
        selected_accounts = list(topic_translations.keys())

    final_keywords = set(BASE_STRONG_PHRASES)
    account_keyword_map = {}
    explore_keywords = set(CRAWL_HELPER_TERMS) | set(BASE_STRONG_PHRASES)

    for account_name in selected_accounts:
        account_phrases = set()

        # Exact account name is a valid conservative phrase.
        account_phrases.add(account_name)
        account_phrases.add(account_name.replace("accounts", "account"))
        account_phrases.add(account_name.replace("accounts", "accounting"))

        translations_for_account = topic_translations.get(account_name, {})

        for language in languages:
            account_words = safe_list(account_words_by_language.get(language))
            topic_words = safe_list(translations_for_account.get(language))

            # English fallback helps because many official statistics pages publish
            # English metadata even outside English-speaking countries.
            if language != "English":
                topic_words += safe_list(translations_for_account.get("English"))

            for topic in topic_words:
                explore_keywords.add(topic)
                for account_word in account_words:
                    account_phrases.add(f"{topic} {account_word}")
                    account_phrases.add(f"{account_word} {topic}")

        # Add exact English account names and configured topic variants for exploration.
        for translations in translations_for_account.values():
            explore_keywords.update(safe_list(translations))

        # Keep phrases long enough to avoid random matches.
        cleaned = {
            " ".join(str(phrase).split())
            for phrase in account_phrases
            if phrase and len(str(phrase).strip()) >= 4
        }

        account_keyword_map[account_name] = sorted(cleaned, key=lambda x: (len(x), x.lower()), reverse=True)
        final_keywords.update(cleaned)

    return {
        "country": country,
        "languages": languages,
        "selected_accounts": selected_accounts,
        "final_keywords": sorted(final_keywords, key=lambda x: (len(x), x.lower()), reverse=True),
        "explore_keywords": sorted(explore_keywords, key=lambda x: (len(x), x.lower()), reverse=True),
        "account_keyword_map": account_keyword_map,
    }


def find_matches(text, keywords):
    """Find keyword matches with normal, accent-insensitive, and compact matching."""
    normalized = normalize_text(text)
    accentless = strip_accents(text)
    compact = compact_text(text)
    matches = []

    for term in keywords:
        term = str(term or "").strip()
        if not term:
            continue

        term_norm = normalize_text(term)
        term_accentless = strip_accents(term)

        if term_norm in normalized or term_accentless in accentless:
            matches.append(term)
            continue

        compact_term = compact_text(term)
        if len(compact_term) >= 5 and compact_term in compact:
            matches.append(term)

    return sorted(set(matches), key=lambda item: (len(item), item.lower()), reverse=True)


def match_accounts(text, profile):
    matched_accounts = []
    matched_terms = []

    for account_name, phrases in profile["account_keyword_map"].items():
        matches = find_matches(text, phrases)
        if matches:
            matched_accounts.append(account_name)
            matched_terms.extend(matches[:5])

    # Base SEEA phrases are not account-specific but are valid for saving.
    base_matches = find_matches(text, BASE_STRONG_PHRASES)
    if base_matches:
        matched_accounts.append("General SEEA/environmental accounts")
        matched_terms.extend(base_matches)

    return sorted(set(matched_accounts)), sorted(set(matched_terms), key=lambda x: (len(x), x.lower()), reverse=True)


def score_text_for_country(text, profile):
    final_matches = find_matches(text, profile["final_keywords"])
    explore_matches = find_matches(text, profile["explore_keywords"])
    accounts, account_terms = match_accounts(text, profile)

    # Strongly prioritize links that meet final saving criteria, but still allow
    # crawling through pages with weaker exploration signals.
    score = len(final_matches) * 10 + len(account_terms) * 15 + len(explore_matches)
    matches = sorted(set(final_matches + account_terms + explore_matches[:5]), key=lambda x: (len(x), x.lower()), reverse=True)
    return score, matches, accounts


def is_relevant_account_text(text, profile):
    accounts, matched_terms = match_accounts(text, profile)
    return bool(matched_terms), matched_terms, accounts


# ---------------------------------------------------------------------------
# Excel hyperlink extraction
# ---------------------------------------------------------------------------
def extract_relationship_targets(zip_file, rel_path):
    if rel_path not in zip_file.namelist():
        return {}

    root = ET.fromstring(zip_file.read(rel_path))
    rel_ns = "{http://schemas.openxmlformats.org/package/2006/relationships}"

    return {
        rel.attrib.get("Id"): rel.attrib.get("Target")
        for rel in root.findall(f"{rel_ns}Relationship")
    }


def get_headers(ws, header_row=2):
    return [cell.value for cell in ws[header_row]]


def extract_country_links_and_accounts_from_workbook(path, sheet_name, config):
    """
    Return:
      country_data[country] = {
          'websites': [...],
          'reported_accounts': [...]
      }
    """
    wb = openpyxl.load_workbook(path, data_only=True)
    ws = wb[sheet_name]

    headers = get_headers(ws, header_row=2)
    header_to_col = {str(header).strip(): idx + 1 for idx, header in enumerate(headers) if header}

    country_col = header_to_col.get("Country or area")
    website_col = header_to_col.get("Website(s)")

    if not country_col or not website_col:
        raise ValueError("Could not find 'Country or area' and/or 'Website(s)' in row 2.")

    configured_accounts = set(config.get("topic_translations", {}).keys())
    account_cols = {
        header: col
        for header, col in header_to_col.items()
        if header in configured_accounts
    }

    row_links = {row: [] for row in range(3, ws.max_row + 1)}
    multi_link_rows = []

    for row in range(3, ws.max_row + 1):
        cell = ws.cell(row, website_col)
        visible_link_count = str(cell.value or "").count("Link")

        if visible_link_count > 1:
            multi_link_rows.append((row, visible_link_count))
        elif cell.hyperlink:
            url = normalize_url(cell.hyperlink.target or cell.hyperlink.location)
            if url:
                row_links[row].append(url)

    # Some multi-link cells are stored as drawing hyperlinks.
    drawing_links = []
    with zipfile.ZipFile(path) as zf:
        drawing_targets = {}
        for rel_path in [
            "xl/drawings/_rels/drawing1.xml.rels",
            "xl/drawings/_rels/drawing2.xml.rels",
            "xl/drawings/_rels/drawing3.xml.rels",
        ]:
            drawing_targets.update(extract_relationship_targets(zf, rel_path))

        for drawing_path in [
            "xl/drawings/drawing1.xml",
            "xl/drawings/drawing2.xml",
            "xl/drawings/drawing3.xml",
        ]:
            if drawing_path not in zf.namelist():
                continue

            root = ET.fromstring(zf.read(drawing_path))
            ns = {
                "xdr": "http://schemas.openxmlformats.org/drawingml/2006/spreadsheetDrawing",
                "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
            }

            for index, anchor in enumerate(root.findall("xdr:twoCellAnchor", ns)):
                row_el = anchor.find("xdr:from/xdr:row", ns)
                hlink = anchor.find(".//a:hlinkClick", ns)
                if row_el is None or hlink is None:
                    continue

                rel_id = hlink.attrib.get(
                    "{http://schemas.openxmlformats.org/officeDocument/2006/relationships}id"
                )
                url = normalize_url(drawing_targets.get(rel_id))
                if url:
                    # XML rows are zero-based.
                    drawing_links.append((int(row_el.text) + 1, index, url))

    drawing_links.sort(key=lambda item: (item[0], item[1]))
    drawing_index = 0
    allocated_urls = set()

    for row, visible_link_count in multi_link_rows:
        allocated = []
        while drawing_index < len(drawing_links) and len(allocated) < visible_link_count:
            _, _, url = drawing_links[drawing_index]
            drawing_index += 1
            if url in allocated_urls:
                continue
            allocated.append(url)
            allocated_urls.add(url)
        row_links[row].extend(allocated)

    country_data = {}
    for row in range(3, ws.max_row + 1):
        country = ws.cell(row, country_col).value
        if not country:
            continue

        reported_accounts = []
        for account_name, col in account_cols.items():
            if cell_is_reported(ws.cell(row, col).value):
                reported_accounts.append(account_name)

        country_data[str(country).strip()] = {
            "websites": list(dict.fromkeys(link for link in row_links[row] if link)),
            "reported_accounts": reported_accounts,
        }

    return country_data


# ---------------------------------------------------------------------------
# Web crawling and downloading
# ---------------------------------------------------------------------------
def make_session():
    session = requests.Session()
    session.headers.update(
        {
            "User-Agent": USER_AGENT,
            "Accept": (
                "text/html,application/xhtml+xml,application/xml,"
                "application/pdf,application/vnd.ms-excel,"
                "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet,"
                "application/vnd.openxmlformats-officedocument.wordprocessingml.document,"
                "*/*;q=0.8"
            ),
        }
    )
    return session


def fetch_url(session, url, stream=False):
    return session.get(
        url,
        timeout=REQUEST_TIMEOUT,
        allow_redirects=True,
        stream=stream,
    )


def get_title_headings_body(soup):
    title = soup.title.get_text(" ", strip=True) if soup.title else ""
    headings = " ".join(
        heading.get_text(" ", strip=True)
        for heading in soup.find_all(re.compile("^h[1-3]$"))
    )
    main = soup.find("main") or soup.find("article") or soup.body or soup
    body_text = main.get_text(" ", strip=True)
    return title, headings, body_text[:12000]


def extract_candidate_links(soup, page_url, source_url, profile):
    candidates = {}

    for tag in soup.find_all("a", href=True):
        raw_url = urllib.parse.urljoin(page_url, tag.get("href"))
        url = normalize_url(raw_url)
        if not url:
            continue

        # Stay on the official site unless the link is directly downloadable.
        if not is_internal_url(url, source_url) and not is_downloadable_url(url):
            continue

        anchor_text = tag.get_text(" ", strip=True)
        parent_text = tag.parent.get_text(" ", strip=True)[:1000] if tag.parent else ""
        context = f"{url} {anchor_text} {parent_text}"
        score, matches, accounts = score_text_for_country(context, profile)

        # For external downloadable URLs, require at least some account signal.
        if not matches and not is_internal_url(url, source_url):
            continue

        key = canonical_url(url)
        candidate = CandidateLink(url, anchor_text, parent_text, score, matches, accounts)
        current = candidates.get(key)
        if current is None or candidate.score > current.score:
            candidates[key] = candidate

    return sorted(candidates.values(), key=lambda item: item.score, reverse=True)


def save_file(country_folder, country, url, response, matched_terms, matched_accounts):
    content_type = response.headers.get("Content-Type", "")
    ext = file_extension(response.url, content_type)
    if ext not in DOWNLOAD_EXTENSIONS:
        ext = ".bin"

    path_name = os.path.basename(urllib.parse.urlparse(response.url).path.rstrip("/"))
    stem = os.path.splitext(path_name)[0] if path_name else ""
    if not stem:
        stem = urllib.parse.urlparse(response.url).netloc

    account_part = matched_accounts[0] if matched_accounts else "SEEA_account"
    keyword_part = matched_terms[0] if matched_terms else "account_file"
    digest = hashlib.sha1(response.url.encode("utf-8")).hexdigest()[:10]

    filename = (
        f"{clean_filename(country, 40)}__"
        f"{clean_filename(account_part, 45)}__"
        f"{clean_filename(keyword_part, 35)}__"
        f"{clean_filename(stem, 70)}__"
        f"{digest}{ext}"
    )

    path = unique_path(country_folder, filename)
    total = 0

    with open(path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 256):
            if not chunk:
                continue
            total += len(chunk)
            if total > MAX_DOWNLOAD_BYTES:
                raise ValueError(f"download exceeded {MAX_DOWNLOAD_BYTES} bytes")
            file.write(chunk)

    return path, os.path.basename(path), total


def save_html(country_folder, country, url, title, html_text, matched_terms, matched_accounts):
    title_part = title or urllib.parse.urlparse(url).path.rstrip("/").split("/")[-1] or "account_page"
    account_part = matched_accounts[0] if matched_accounts else "SEEA_account"
    keyword_part = matched_terms[0] if matched_terms else "account"
    digest = hashlib.sha1(url.encode("utf-8")).hexdigest()[:10]

    filename = (
        f"{clean_filename(country, 40)}__"
        f"{clean_filename(account_part, 45)}__"
        f"{clean_filename(keyword_part, 35)}__"
        f"{clean_filename(title_part, 70)}__"
        f"{digest}.html"
    )
    path = unique_path(country_folder, filename)
    with open(path, "w", encoding="utf-8") as file:
        file.write(html_text)
    return path, os.path.basename(path)


def crawl_country(session, country, source_website, profile, args):
    country_folder = Path(args.output_dir) / clean_filename(country, 100)
    country_folder.mkdir(parents=True, exist_ok=True)

    queue = []
    sequence = 0
    heappush(queue, (0, sequence, source_website, 0, source_website, ""))

    visited = set()
    saved_urls = set()
    visited_count = 0
    download_rows = []
    search_rows = []

    while queue and visited_count < args.max_pages_per_country:
        _, _, url, depth, discovered_from, link_context = heappop(queue)
        key = canonical_url(url)
        if key in visited or depth > args.max_depth:
            continue
        visited.add(key)

        page_title = ""
        candidate_count = 0
        relevant_count = 0
        saved_count = 0
        status = "visited"
        error_message = ""

        try:
            response = fetch_url(session, url, stream=is_downloadable_url(url))
            response.raise_for_status()
            final_url = normalize_url(response.url) or url
            content_type = response.headers.get("Content-Type", "")

            if is_downloadable_url(final_url, content_type):
                context = f"{final_url} {discovered_from} {link_context} {content_type}"
                relevant, matched_terms, matched_accounts = is_relevant_account_text(context, profile)

                if relevant and canonical_url(final_url) not in saved_urls:
                    path, name, size = save_file(
                        country_folder,
                        country,
                        final_url,
                        response,
                        matched_terms,
                        matched_accounts,
                    )
                    saved_urls.add(canonical_url(final_url))
                    saved_count += 1
                    relevant_count += 1

                    download_rows.append(
                        {
                            "country": country,
                            "source_website": source_website,
                            "discovered_from_url": discovered_from,
                            "saved_url": final_url,
                            "file_or_page_name": name,
                            "saved_path": path,
                            "content_type": content_type,
                            "file_type": file_extension(final_url, content_type),
                            "matched_account": "; ".join(matched_accounts),
                            "matched_keyword": "; ".join(matched_terms),
                            "match_reason": "downloadable file matched country-language account keyword from JSON config",
                            "search_depth": depth,
                            "download_status": "saved",
                            "error_message": "",
                        }
                    )
                else:
                    response.close()
                status = "download_checked"

            else:
                content = response.content[:MAX_HTML_BYTES]
                html_text = content.decode(response.encoding or "utf-8", errors="replace")
                soup = BeautifulSoup(html_text, "html.parser")
                page_title, headings, body_text = get_title_headings_body(soup)

                primary_context = f"{final_url} {page_title} {headings}"
                page_relevant, page_terms, page_accounts = is_relevant_account_text(primary_context, profile)

                # Saving HTML is optional. The default is False because your main aim
                # is to download account files, not collect broad webpages.
                if args.save_html_pages and page_relevant and canonical_url(final_url) not in saved_urls:
                    path, name = save_html(
                        country_folder,
                        country,
                        final_url,
                        page_title,
                        html_text,
                        page_terms,
                        page_accounts,
                    )
                    saved_urls.add(canonical_url(final_url))
                    saved_count += 1
                    download_rows.append(
                        {
                            "country": country,
                            "source_website": source_website,
                            "discovered_from_url": discovered_from,
                            "saved_url": final_url,
                            "file_or_page_name": name,
                            "saved_path": path,
                            "content_type": content_type,
                            "file_type": ".html",
                            "matched_account": "; ".join(page_accounts),
                            "matched_keyword": "; ".join(page_terms),
                            "match_reason": "HTML URL/title/headings matched country-language account keyword from JSON config",
                            "search_depth": depth,
                            "download_status": "saved",
                            "error_message": "",
                        }
                    )

                candidates = extract_candidate_links(soup, final_url, source_website, profile)
                candidate_count = len(candidates)
                relevant_count = sum(1 for candidate in candidates if candidate.matched_terms)

                if depth < args.max_depth:
                    followable = [
                        candidate
                        for candidate in candidates
                        if canonical_url(candidate.url) not in visited
                        and (
                            is_internal_url(candidate.url, source_website)
                            or is_downloadable_url(candidate.url)
                        )
                    ]

                    for candidate in followable[: args.max_links_per_page]:
                        if candidate.score >= args.min_follow_score or depth == 0:
                            sequence += 1
                            priority = -(candidate.score * 10 - depth)
                            context = f"{candidate.anchor_text} {candidate.nearby_text}"
                            heappush(queue, (priority, sequence, candidate.url, depth + 1, final_url, context))

            visited_count += 1

        except Exception as exc:
            status = "error"
            error_message = f"{type(exc).__name__}: {exc}"

        search_rows.append(
            {
                "country": country,
                "source_website": source_website,
                "visited_url": url,
                "page_title": page_title,
                "search_depth": depth,
                "candidate_links_found": candidate_count,
                "relevant_links_found": relevant_count,
                "saved_items_count": saved_count,
                "status": status,
                "error_message": error_message,
            }
        )

        if args.delay_seconds:
            time.sleep(args.delay_seconds)

    return download_rows, search_rows


# ---------------------------------------------------------------------------
# CSV logging
# ---------------------------------------------------------------------------
def append_csv(path, rows, fieldnames):
    if not rows:
        return

    write_header = not os.path.exists(path) or os.path.getsize(path) == 0
    with open(path, "a", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        if write_header:
            writer.writeheader()
        writer.writerows(rows)
        file.flush()


def write_keyword_dictionary(path, profiles):
    rows = []
    for profile in profiles:
        country = profile["country"]
        languages = "; ".join(profile["languages"])
        for account_name, phrases in profile["account_keyword_map"].items():
            for phrase in phrases:
                rows.append(
                    {
                        "country": country,
                        "languages": languages,
                        "account_name": account_name,
                        "keyword_phrase": phrase,
                    }
                )

    fieldnames = ["country", "languages", "account_name", "keyword_phrase"]
    with open(path, "w", newline="", encoding="utf-8-sig") as file:
        writer = csv.DictWriter(file, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


# ---------------------------------------------------------------------------
# Arguments and main run
# ---------------------------------------------------------------------------
def parse_args():
    parser = argparse.ArgumentParser(
        description="Download SEEA account files using country-language keywords from a JSON configuration."
    )
    parser.add_argument("--working-dir", help="Optional working directory. If set, the script changes into it first.")
    parser.add_argument("--source-xlsx", default=DEFAULT_SOURCE_XLSX)
    parser.add_argument("--config-json", default=DEFAULT_CONFIG_JSON)
    parser.add_argument("--sheet-name", default=DEFAULT_SHEET_NAME)
    parser.add_argument("--output-dir", default=DEFAULT_OUTPUT_DIR)
    parser.add_argument("--countries", nargs="*", help="Optional country names to process. Default: all countries.")
    parser.add_argument("--start-after", help="Skip countries through this country name, then process the rest.")
    parser.add_argument("--limit-countries", type=int, help="Process only the first N selected countries.")
    parser.add_argument("--max-depth", type=int, default=DEFAULT_MAX_DEPTH)
    parser.add_argument("--max-pages-per-country", type=int, default=DEFAULT_MAX_PAGES_PER_COUNTRY)
    parser.add_argument("--max-links-per-page", type=int, default=DEFAULT_MAX_LINKS_PER_PAGE)
    parser.add_argument("--min-follow-score", type=int, default=2)
    parser.add_argument("--delay-seconds", type=float, default=DEFAULT_DELAY_SECONDS)
    parser.add_argument(
        "--all-config-accounts",
        action="store_true",
        help="Use all accounts from the JSON config instead of only accounts reported in the workbook row.",
    )
    parser.add_argument(
        "--save-html-pages",
        action="store_true",
        help="Also save relevant HTML account pages. Default is to download files only.",
    )
    return parser.parse_args()


def main():
    args = parse_args()

    if args.working_dir:
        os.chdir(args.working_dir)
    print("Current working directory:", os.getcwd())

    Path(args.output_dir).mkdir(parents=True, exist_ok=True)

    config = load_keyword_config(args.config_json)
    country_data = extract_country_links_and_accounts_from_workbook(
        args.source_xlsx,
        args.sheet_name,
        config,
    )

    if args.countries:
        wanted = {country.lower() for country in args.countries}
        country_data = {
            country: data
            for country, data in country_data.items()
            if country.lower() in wanted
        }

    if args.start_after:
        selected = {}
        skip = True
        for country, data in country_data.items():
            if skip:
                if country.lower() == args.start_after.lower():
                    skip = False
                continue
            selected[country] = data
        country_data = selected

    if args.limit_countries is not None:
        country_data = dict(list(country_data.items())[: args.limit_countries])

    download_fields = [
        "country", "source_website", "discovered_from_url", "saved_url",
        "file_or_page_name", "saved_path", "content_type", "file_type",
        "matched_account", "matched_keyword", "match_reason", "search_depth",
        "download_status", "error_message",
    ]

    search_fields = [
        "country", "source_website", "visited_url", "page_title", "search_depth",
        "candidate_links_found", "relevant_links_found", "saved_items_count",
        "status", "error_message",
    ]

    session = make_session()
    profiles_for_log = []
    total_saved = 0
    total_search_rows = 0

    for country, data in country_data.items():
        websites = data.get("websites", [])
        reported_accounts = data.get("reported_accounts", [])

        profile = generate_country_keyword_profile(
            country=country,
            account_names=reported_accounts,
            config=config,
            use_reported_accounts=not args.all_config_accounts,
        )
        profiles_for_log.append(profile)

        country_folder = Path(args.output_dir) / clean_filename(country, 100)
        country_folder.mkdir(parents=True, exist_ok=True)

        if not websites:
            missing_rows = [
                {
                    "country": country,
                    "source_website": "",
                    "visited_url": "",
                    "page_title": "",
                    "search_depth": 0,
                    "candidate_links_found": 0,
                    "relevant_links_found": 0,
                    "saved_items_count": 0,
                    "status": "missing_hyperlink",
                    "error_message": "",
                }
            ]
            append_csv(SEARCH_LOG, missing_rows, search_fields)
            total_search_rows += len(missing_rows)
            print(f"{country}: no Website(s) hyperlink")
            continue

        print(
            f"{country}: {len(websites)} starting website(s); "
            f"{len(profile['selected_accounts'])} account type(s) searched; "
            f"languages: {', '.join(profile['languages'])}"
        )

        for website in websites:
            download_rows, search_rows = crawl_country(session, country, website, profile, args)
            append_csv(DOWNLOAD_LOG, download_rows, download_fields)
            append_csv(SEARCH_LOG, search_rows, search_fields)

            total_saved += len(download_rows)
            total_search_rows += len(search_rows)

            print(
                f"  {website} -> saved {len(download_rows)} item(s), "
                f"visited {len(search_rows)} page(s)"
            )

    write_keyword_dictionary(KEYWORD_DICTIONARY_LOG, profiles_for_log)

    print("")
    print(f"Done. Saved {total_saved} relevant file(s) into {args.output_dir}")
    print(f"Wrote {DOWNLOAD_LOG}")
    print(f"Wrote {SEARCH_LOG}")
    print(f"Wrote {KEYWORD_DICTIONARY_LOG}")
    print(f"Search log rows: {total_search_rows}")


if __name__ == "__main__":
    main()
